# 🛡️ IBM AML Stateful Behavioral Fraud Dataset
## *An Exploratory Analysis of 12 Million Financial Transactions*

---

## 📌 Executive Summary & Introduction

Anti-Money Laundering (AML) and financial fraud detection represent some of the most critical challenges in modern computational finance. Financial institutions process billions of transactions daily, facing two fundamental technical hurdles:

1. **Extreme Class Imbalance**: Legitimate transactions dwarf fraudulent ones (often $< 0.1\%$ positive cases).
2. **Stateful Behavioral Dynamics**: Static, transaction-level snapshots (e.g. single transaction amount) fail to capture anomalous behavior. Fraud is inherently **behavioral** — a $\$5,000$ transfer is normal for a corporate treasury account but highly suspicious for an account that has only ever transferred $\$10$.

### The Pitfall of Traditional Public Datasets
Most publicly available fraud detection benchmarks suffer from subtle yet catastrophic **data leakage**:
- Global aggregates (e.g. mean account spending over the entire multi-year dataset) leak future information into historical predictions.
- Random train/test splits allow models to "see into the future," resulting in artificially inflated test metrics ($99.9\%$ AUC) that crash upon real-world deployment.

### Motivation for Stateful Online Features & Dual Evaluation
This dataset bridges the gap between academic benchmarks and production financial infrastructure:
- **Stateful Online Feature Engineering**: Features are computed sequentially using an `AccountState` object that updates **only after** a transaction's features are generated. No future data is ever visible.
- **Dual Evaluation Protocol**: Provides both a **Benchmark Split** (stratified random for standard model comparison) and a **Chronological Split** (strict temporal ordering to evaluate real-world production decay and temporal leakage).

---


## 🌟 Why This Dataset Is Different

This dataset provides a production-grade representation of real-world streaming ingestion gateways. Below are the key architectural pillars that distinguish it from standard tabular datasets:

### 1.1 Stateful Behavioral Features
Features are computed sequentially using historical account state rather than static transaction snapshots. Every transaction is processed in strict chronological order per account, ensuring that feature vectors depend exclusively on information available *before* the transaction occurred.

### 1.2 Online Feature Generation
The dataset models a real-time event-streaming pipeline (mimicking systems built on Kafka or Flink). Sender account profiles (`AccountState`) continuously track spending habits, velocity, and network relationships dynamically as events arrive.

### 1.3 Leakage-Free Feature Engineering
Target leakage is mathematically eliminated. By strictly separating feature computation ($t < 	ext{current\_time}$) from state updates ($t \ge 	ext{current\_time}$), the pipeline ensures zero future data contamination.

### 1.4 Dual Evaluation Strategy
The dataset includes two distinct evaluation protocols:
- **Benchmark Split (Stratified Random 70/15/15)**: Maintains uniform fraud distribution across splits; ideal for model architecture comparison.
- **Chronological Split (Temporal 70/15/15)**: Partitioned strictly by time; tests model stability under temporal drift and realistic deployment conditions.

### 1.5 Production Validation
The entire dataset passed a 100% rigorous automated validation suite covering schema consistency, null-value checks, non-negative timestamp diffs, label integrity, and Welford algorithm z-score verification.

### 1.6 Scale & Realism
- **12,002,394** total transactions
- **8,746** confirmed laundering/fraud cases
- **15** engineered behavioral and temporal features
- Severe real-world class imbalance (**~0.073% fraud ratio**)

### 1.7 Research & Industry Applications
- **Financial Crime & AML Research**: Benchmarking graph and tabular anomaly detection algorithms.
- **Temporal Machine Learning**: Studying concept drift and temporal feature decay.
- **Explainable AI (XAI)**: Analyzing behavioral feature importance (e.g. z-score deviations vs velocity).
- **Production Pipeline Benchmarking**: Comparing classical gradient boosters (XGBoost, LightGBM, CatBoost) against deep neural architectures.

> **Summary**: Through stateful, leakage-free feature engineering and dual evaluation protocols, this dataset provides a realistic sandbox for developing robust fraud detection systems that translate directly to production deployment.


## 🛠️ Environment Setup & Data Loading

We begin by setting up imports, styling preferences, and helper functions to load dataset partitions smoothly across local and Kaggle environments.


In [ ]:
import os
import sys
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.titlesize'] = 14

# Palette
PALETTE = {'Legitimate': '#2b5c8f', 'Fraud': '#d9534f', 'Train': '#3366cc', 'Valid': '#ff9900', 'Test': '#109618'}
CUSTOM_COLORS = ['#2b5c8f', '#d9534f', '#428bca', '#5cb85c', '#f0ad4e', '#5bc0de', '#aa66cc']

def locate_data_dir():
    """Find dataset outputs directory across Kaggle and local environments."""
    candidates = [
        Path('/kaggle/input/ibm-aml-stateful-behavioral-dataset/outputs'),
        Path('/kaggle/input/ibm-aml-stateful-behavioral-features/outputs'),
        Path('../dataset/outputs'),
        Path('kaggle/dataset/outputs'),
        Path('C:/Projects/fintech-pipeline/kaggle/dataset/outputs'),
    ]
    for c in candidates:
        if c.exists() and (c / 'training_dataset_hi_li_small.parquet').exists():
            return c
    # Fallback search
    matches = glob.glob('**/training_dataset_hi_li_small.parquet', recursive=True)
    if matches:
        return Path(matches[0]).parent
    raise FileNotFoundError("Dataset parquet files not found. Ensure dataset is attached.")

DATA_DIR = locate_data_dir()
CHRONO_DIR = DATA_DIR.parent / 'chronological' if (DATA_DIR.parent / 'chronological').exists() else DATA_DIR

print(f"[OK] Data directory located at: {DATA_DIR}")
print(f"[OK] Chronological directory located at: {CHRONO_DIR}")


## 📊 Section 2: Dataset Overview

Let's load the full combined dataset (`training_dataset_hi_li_small.parquet`) and inspect key metrics, memory footprint, and data types.


In [ ]:
# Load Full Combined Dataset
full_path = DATA_DIR / 'training_dataset_hi_li_small.parquet'
df_full = pd.read_parquet(full_path)

total_rows = len(df_full)
fraud_count = int(df_full['is_fraud'].sum())
legit_count = total_rows - fraud_count
fraud_ratio = df_full['is_fraud'].mean() * 100
num_features = len(df_full.columns) - 1
mem_usage_mb = df_full.memory_usage(deep=True).sum() / (1024 * 1024)

# Overview Dashboard Table
overview_data = {
    "Metric": [
        "Total Transactions",
        "Legitimate Transactions",
        "Fraudulent Transactions",
        "Fraud Ratio (%)",
        "Engineered Features",
        "Target Column",
        "Memory Usage (MB)",
        "Data Format"
    ],
    "Value": [
        f"{total_rows:,}",
        f"{legit_count:,}",
        f"{fraud_count:,}",
        f"{fraud_ratio:.4f}%",
        f"{num_features}",
        "is_fraud",
        f"{mem_usage_mb:.2f} MB",
        "Apache Parquet"
    ]
}
overview_df = pd.DataFrame(overview_data)
display(overview_df)

print("--- Dataset Preview (First 5 Rows) ---")
display(df_full.head())


## 📋 Section 3: Dataset Schema

The dataset contains 16 columns: 15 stateful engineered behavioral/temporal features and 1 ground-truth target label (`is_fraud`).


In [ ]:
schema_data = [
    {"Feature Name": "amount", "Data Type": "float64", "Category": "Monetary", "Description": "Transaction payment amount in USD equivalent", "Target?": "No"},
    {"Feature Name": "spending_deviation_score", "Data Type": "float64", "Category": "Monetary / Behavioral", "Description": "Welford Z-Score of amount vs account's historical spending mean & std (std clamped min 1.0)", "Target?": "No"},
    {"Feature Name": "hour", "Data Type": "int64", "Category": "Temporal", "Description": "Hour of transaction execution (0 to 23)", "Target?": "No"},
    {"Feature Name": "day_of_week", "Data Type": "int64", "Category": "Temporal", "Description": "Day of the week (0 = Monday, 6 = Sunday)", "Target?": "No"},
    {"Feature Name": "month", "Data Type": "int64", "Category": "Temporal", "Description": "Month of transaction execution (1 to 12)", "Target?": "No"},
    {"Feature Name": "is_weekend", "Data Type": "int64", "Category": "Temporal", "Description": "Binary flag (1 if Saturday or Sunday, else 0)", "Target?": "No"},
    {"Feature Name": "time_since_last_transaction", "Data Type": "float64", "Category": "Temporal / Behavioral", "Description": "Elapsed seconds since account's prior transaction (0.0 for first transaction)", "Target?": "No"},
    {"Feature Name": "velocity_score", "Data Type": "int64", "Category": "Temporal / Behavioral", "Description": "Lifetime cumulative count of prior transactions executed by sender account", "Target?": "No"},
    {"Feature Name": "is_first_transaction", "Data Type": "Temporal / Behavioral", "Category": "Behavioral", "Description": "Binary flag (1 if account's first observed transaction, else 0)", "Target?": "No"},
    {"Feature Name": "is_new_receiver", "Data Type": "int64", "Category": "Relationship", "Description": "Binary flag (1 if beneficiary receiver account is seen for first time by sender)", "Target?": "No"},
    {"Feature Name": "is_new_bank", "Data Type": "int64", "Category": "Relationship", "Description": "Binary flag (1 if target bank ID is new for this sender account)", "Target?": "No"},
    {"Feature Name": "is_new_payment_format", "Data Type": "int64", "Category": "Relationship", "Description": "Binary flag (1 if payment channel is new for this sender account)", "Target?": "No"},
    {"Feature Name": "is_cross_bank_transfer", "Data Type": "int64", "Category": "Relationship", "Description": "Binary flag (1 if From Bank != To Bank, inter-bank transfer)", "Target?": "No"},
    {"Feature Name": "is_cross_currency_transfer", "Data Type": "int64", "Category": "Relationship", "Description": "Binary flag (1 if Payment Currency != Receiving Currency, FX transfer)", "Target?": "No"},
    {"Feature Name": "payment_channel", "Data Type": "string", "Category": "Transaction Channel", "Description": "Payment method format (ACH, Wire, Credit Card, Cheque, Cash, Bitcoin, Reinvestment)", "Target?": "No"},
    {"Feature Name": "is_fraud", "Data Type": "int64", "Category": "Ground Truth Target", "Description": "Ground-truth target label (1 = Laundering / Fraud, 0 = Legitimate)", "Target?": "YES"}
]

schema_df = pd.DataFrame(schema_data)
display(schema_df)


## 🛡️ Section 4: Data Quality Assessment

Before performing exploratory analysis, we execute a thorough data quality audit:
1. **Null & Missing Value Audit**: Verifying zero missing values across all columns.
2. **Feature Range & Boundary Validation**: Checking bounds for timestamps, hours, and amounts.
3. **Duplicate Analysis**: Inspecting identical rows and explaining structural feature collisions.


In [ ]:
# 1. Null Value Audit
null_series = df_full.isnull().sum()
null_pct = (df_full.isnull().sum() / len(df_full)) * 100
quality_df = pd.DataFrame({"Missing Count": null_series, "Missing (%)": null_pct})

# 2. Unique Values & Range Checks
min_vals = []
max_vals = []
unique_counts = []

for col in df_full.columns:
    unique_counts.append(df_full[col].nunique())
    if pd.api.types.is_numeric_dtype(df_full[col]):
        min_vals.append(round(df_full[col].min(), 4))
        max_vals.append(round(df_full[col].max(), 4))
    else:
        min_vals.append("N/A")
        max_vals.append("N/A")

quality_df["Unique Values"] = unique_counts
quality_df["Min Value"] = min_vals
quality_df["Max Value"] = max_vals

display(quality_df)

# 3. Duplicate Feature Vector Analysis
dup_count = df_full.duplicated().sum()
dup_pct = (dup_count / len(df_full)) * 100

print("--- Duplicate Feature Vector Check ---")
print(f"Total Duplicate Rows: {dup_count:,} ({dup_pct:.4f}%)")
print("Note on Structural Feature Collisions: The dataset deliberately excludes raw Account IDs to prevent memorization.")
print("When two distinct accounts execute their first transaction with identical amounts, channels, and hours,")
print("their 15-feature vectors are mathematically identical. This is an expected structural collision, not a dataset error.")


## 📈 Section 5: Exploratory Data Analysis (EDA)

We explore the macro transaction properties: class imbalance, transaction amounts, payment channel preferences, and temporal dynamics.


In [ ]:
# Fig 1: Class Distribution & Amount Analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Class Distribution (Bar Plot)
sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=[legit_count, fraud_count], palette=['#2b5c8f', '#d9534f'], ax=axes[0, 0])
axes[0, 0].set_title('Transaction Class Distribution (Severe Imbalance)')
axes[0, 0].set_ylabel('Transaction Count (Log Scale)')
axes[0, 0].set_yscale('log')
for i, v in enumerate([legit_count, fraud_count]):
    axes[0, 0].text(i, v * 1.2, f"{v:,}\n({v/total_rows*100:.4f}%)", ha='center', fontweight='bold')

# 2. Transaction Amount Distribution (Log Scale)
sns.histplot(df_full['amount'], bins=50, log_scale=True, color='#2b5c8f', ax=axes[0, 1], kde=True)
axes[0, 1].set_title('Transaction Amount Distribution (Log-Scaled USD)')
axes[0, 1].set_xlabel('Amount Paid ($ USD, Log Scale)')
axes[0, 1].set_ylabel('Frequency Count')

# 3. Amount Comparison: Fraud vs Legitimate (Boxplot)
sns.boxplot(data=df_full, x='is_fraud', y='amount', palette=['#2b5c8f', '#d9534f'], ax=axes[1, 0])
axes[1, 0].set_title('Transaction Amount by Class (Log Scale Boxplot)')
axes[1, 0].set_yscale('log')
axes[1, 0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Amount Paid ($ USD)')

# 4. Fraud Ratio by Payment Channel
channel_stats = df_full.groupby('payment_channel')['is_fraud'].agg(['count', 'sum', 'mean']).reset_index()
channel_stats['fraud_rate_pct'] = channel_stats['mean'] * 100
channel_stats = channel_stats.sort_values(by='fraud_rate_pct', ascending=False)

sns.barplot(data=channel_stats, x='payment_channel', y='fraud_rate_pct', palette='Reds_r', ax=axes[1, 1])
axes[1, 1].set_title('Fraud Rate (%) by Payment Channel')
axes[1, 1].set_xlabel('Payment Channel')
axes[1, 1].set_ylabel('Fraud Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=30)
for i, row in enumerate(channel_stats.itertuples()):
    axes[1, 1].text(i, row.fraud_rate_pct + 0.005, f"{row.fraud_rate_pct:.3f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Table: Amount Summary Statistics by Class
amount_summary = df_full.groupby('is_fraud')['amount'].describe().rename(index={0: 'Legitimate', 1: 'Fraud'})
display(amount_summary)


### ⏰ Temporal Patterns & Hourly Fraud Rates

Money laundering and automated financial fraud often follow non-standard temporal distributions compared to regular consumer banking activity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly Volume
hourly_counts = df_full.groupby(['hour', 'is_fraud']).size().unstack(fill_value=0)
hourly_counts.columns = ['Legitimate', 'Fraud']

hourly_counts['Legitimate'].plot(kind='bar', color='#2b5c8f', ax=axes[0], width=0.8)
axes[0].set_title('Transaction Volume by Hour of Day')
axes[0].set_xlabel('Hour of Day (0–23)')
axes[0].set_ylabel('Transaction Count')

# Hourly Fraud Rate (%)
hourly_fraud_rate = df_full.groupby('hour')['is_fraud'].mean() * 100
hourly_fraud_rate.plot(kind='line', marker='o', color='#d9534f', linewidth=2.5, ax=axes[1])
axes[1].set_title('Fraud Rate (%) by Hour of Day')
axes[1].set_xlabel('Hour of Day (0–23)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 🔬 Section 6: Behavioral Feature Exploration

We examine the core stateful behavioral features extracted by `AccountState`:
1. `spending_deviation_score`: Welford Z-score of transaction amount relative to historical spending.
2. `velocity_score`: Lifetime cumulative transaction count.
3. `time_since_last_transaction`: Elapsed seconds since previous event.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Spending Deviation Score (Filtered for visualization)
sns.kdeplot(data=df_full, x='spending_deviation_score', hue='is_fraud', common_norm=False,
            palette=['#2b5c8f', '#d9534f'], ax=axes[0, 0], clip=(-5, 15))
axes[0, 0].set_title('Spending Deviation Score (Z-Score KDE)')
axes[0, 0].set_xlabel('Spending Deviation Score (Z-Score)')
axes[0, 0].set_ylabel('Density')

# 2. Velocity Score Distribution (Log Scale)
sns.histplot(data=df_full, x='velocity_score', hue='is_fraud', log_scale=True,
             palette=['#2b5c8f', '#d9534f'], ax=axes[0, 1], bins=40, element='step')
axes[0, 1].set_title('Velocity Score (Lifetime Prior Tx Count, Log Scale)')
axes[0, 1].set_xlabel('Velocity Score')

# 3. Time Since Last Transaction (Seconds, Log Scale)
sns.boxplot(data=df_full[df_full['time_since_last_transaction'] > 0], x='is_fraud', y='time_since_last_transaction',
            palette=['#2b5c8f', '#d9534f'], ax=axes[1, 0])
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('Time Since Last Transaction (Seconds, Log Scale)')
axes[1, 0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

# 4. Relationship Binary Flags (New Receiver, New Bank, Cross Currency)
rel_cols = ['is_new_receiver', 'is_new_bank', 'is_new_payment_format', 'is_cross_bank_transfer', 'is_cross_currency_transfer']
rel_rates = df_full.groupby('is_fraud')[rel_cols].mean().T
rel_rates.columns = ['Legitimate', 'Fraud']

rel_rates.plot(kind='barh', ax=axes[1, 1], color=['#2b5c8f', '#d9534f'], width=0.7)
axes[1, 1].set_title('Relationship & Graph Feature Frequencies')
axes[1, 1].set_xlabel('Proportion (0.0 to 1.0)')

plt.tight_layout()
plt.show()

# Table: Summary Statistics for Behavioral Features
beh_cols = ['spending_deviation_score', 'velocity_score', 'time_since_last_transaction']
display(df_full.groupby('is_fraud')[beh_cols].describe().T)


## 🔗 Section 7: Correlation Analysis

We calculate the pairwise correlation matrix across numeric features to assess feature independence and identify target correlations.


In [ ]:
numeric_cols = [col for col in df_full.columns if pd.api.types.is_numeric_dtype(df_full[col])]
corr_matrix = df_full[numeric_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Pairwise Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

# Top Target Correlations
fraud_corr = corr_matrix['is_fraud'].drop('is_fraud').sort_values(ascending=False)
print("--- Top Feature Correlations with Target (is_fraud) ---")
display(pd.DataFrame({'Correlation with is_fraud': fraud_corr}))


## ⚖️ Section 8: Benchmark vs Chronological Evaluation

The dataset provides two distinct evaluation protocols. Below we load both split sets and compare their partition sizes, fraud ratios, and timestamp bounds.


In [ ]:
# Load Benchmark Splits
train_b = pd.read_parquet(DATA_DIR / 'train.parquet')
valid_b = pd.read_parquet(DATA_DIR / 'valid.parquet')
test_b  = pd.read_parquet(DATA_DIR / 'test.parquet')

# Load Chronological Splits
train_c = pd.read_parquet(CHRONO_DIR / 'train.parquet')
valid_c = pd.read_parquet(CHRONO_DIR / 'valid.parquet')
test_c  = pd.read_parquet(CHRONO_DIR / 'test.parquet')

split_comparison_data = [
    {"Strategy": "Benchmark (Random)", "Partition": "Train", "Rows": f"{len(train_b):,}", "Fraud Ratio (%)": f"{train_b['is_fraud'].mean()*100:.4f}%", "Timestamp Min": "Full Range", "Timestamp Max": "Full Range"},
    {"Strategy": "Benchmark (Random)", "Partition": "Valid", "Rows": f"{len(valid_b):,}", "Fraud Ratio (%)": f"{valid_b['is_fraud'].mean()*100:.4f}%", "Timestamp Min": "Full Range", "Timestamp Max": "Full Range"},
    {"Strategy": "Benchmark (Random)", "Partition": "Test",  "Rows": f"{len(test_b):,}",  "Fraud Ratio (%)": f"{test_b['is_fraud'].mean()*100:.4f}%",  "Timestamp Min": "Full Range", "Timestamp Max": "Full Range"},
    {"Strategy": "Chronological (Temporal)", "Partition": "Train", "Rows": f"{len(train_c):,}", "Fraud Ratio (%)": f"{train_c['is_fraud'].mean()*100:.4f}%", "Timestamp Min": "Month 9, Hour 0", "Timestamp Max": "Month 9, Hour 16"},
    {"Strategy": "Chronological (Temporal)", "Partition": "Valid", "Rows": f"{len(valid_c):,}", "Fraud Ratio (%)": f"{valid_c['is_fraud'].mean()*100:.4f}%", "Timestamp Min": "Month 9, Hour 16", "Timestamp Max": "Month 9, Hour 20"},
    {"Strategy": "Chronological (Temporal)", "Partition": "Test",  "Rows": f"{len(test_c):,}",  "Fraud Ratio (%)": f"{test_c['is_fraud'].mean()*100:.4f}%",  "Timestamp Min": "Month 9, Hour 20", "Timestamp Max": "Month 9, Hour 23"},
]

comp_df = pd.DataFrame(split_comparison_data)
display(comp_df)

# Visualizing Fraud Ratio Variation Across Strategies
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
width = 0.35

b_ratios = [train_b['is_fraud'].mean()*100, valid_b['is_fraud'].mean()*100, test_b['is_fraud'].mean()*100]
c_ratios = [train_c['is_fraud'].mean()*100, valid_c['is_fraud'].mean()*100, test_c['is_fraud'].mean()*100]

rects1 = ax.bar(x - width/2, b_ratios, width, label='Benchmark (Stratified)', color='#2b5c8f')
rects2 = ax.bar(x + width/2, c_ratios, width, label='Chronological (Temporal)', color='#d9534f')

ax.set_title('Fraud Ratio (%) Comparison: Benchmark vs Chronological Splits')
ax.set_xticks(x)
ax.set_xticklabels(['Train (70%)', 'Valid (15%)', 'Test (15%)'])
ax.set_ylabel('Fraud Ratio (%)')
ax.legend()
plt.tight_layout()
plt.show()


## 🏗️ Section 9: Stateful Feature Engineering Architecture

The streaming feature engineering architecture relies on `FeaturePipeline` and `AccountState`.

### Stream Processing Workflow

```
┌─────────────────────────────────────────────────────────────┐
│ 1. Read Streaming Transaction (Account A -> Account B)      │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. Read CURRENT AccountState(A)                             │
│    (State contains information ONLY for t < current_time)   │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. Compute Features (Z-Score, Velocity, New Relationship)   │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. Output Feature Vector to Parquet File                    │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 5. Update AccountState(A) with Current Transaction           │
└─────────────────────────────────────────────────────────────┘
```

### Online Welford Algorithm
Online mean and variance are computed using Welford's algorithm:

$$\Delta = x_n - \mu_{n-1}$$
$$\mu_n = \mu_{n-1} + rac{\Delta}{n}$$
$$M_{2,n} = M_{2,n-1} + \Delta \cdot (x_n - \mu_n)$$
$$\sigma_n^2 = rac{M_{2,n}}{n - 1}$$

Standard deviation is clamped at minimum $1.0$ to prevent division by near-zero variance.


## 📋 Section 10: Dataset Validation Summary

The dataset has passed an automated technical audit.

| Validation Check | Target Requirement | Audit Result | Status |
| :--- | :--- | :---: | :---: |
| **Schema Uniformity** | 16 identical columns across all 7 parquet files | 16 / 16 Columns Match | ✅ PASS |
| **Null Value Enforcement** | 0 missing values across all engineered features | 0 Nulls | ✅ PASS |
| **Timestamp Integrity** | `time_since_last_transaction` $\ge 0.00	ext{ s}$ | Min = 0.00 s | ✅ PASS |
| **Label Integrity** | Ground truth label `is_fraud` $\in \{0, 1\}$ | Only {0, 1} | ✅ PASS |
| **Categorical Integrity** | Clean string payment formats without nulls | 7 Clean Categories | ✅ PASS |
| **Split Count Integrity** | Split row sum equals full dataset | 12,002,394 = 12,002,394 | ✅ PASS |
| **Reproducibility** | SHA256 file checksums recorded in metadata | Verified | ✅ PASS |


## ⚠️ Section 11: Dataset Limitations

While this dataset provides a significant advancement in leakage-free stateful feature engineering, researchers should keep in mind the following constraints:

1. **Synthetic Foundation**: Derived from IBM's synthetic AML transaction generator (`HI-Small` and `LI-Small`).
2. **Severe Class Imbalance**: Extreme positive class scarcity (~0.073% fraud) requires specialized metrics (Precision-Recall AUC, F1 at low false-positive rates) rather than standard accuracy or ROC-AUC.
3. **Anonymized Features**: Raw account identifiers and text descriptions are omitted to focus on generalizable behavioral patterns.


## 🎯 Section 12: Conclusion & Next Steps

This notebook demonstrated the technical quality, stateful feature engineering architecture, and dual evaluation protocols of the **IBM AML Stateful Behavioral Fraud Dataset**.

### Key Takeaways:
- **12,002,394** transactions processed through an online stateful feature engineering pipeline.
- **Zero target leakage**: Features depend strictly on historical state prior to each transaction event.
- **Dual Evaluation Protocols**: Benchmark (stratified random) for baseline ML modeling and Chronological (temporal) for realistic production backtesting.

---
*Created by the Fintech Pipeline Engineering Team.*
